# 모듈 0. 코스 소개와 환경 설정

**AI로 EDGAR 공시를 수집해 JP Morgan 구조화 상품을 해부하고 신상품을 설계한다.**

이 코스에서 AI는 별도 API 없이 **Claude Desktop** 하나만 씁니다. 역할은 세 가지입니다.

| 역할 | 어떻게 | 해당 모듈 |
|---|---|---|
| **코딩 파트너** | Code 탭에서 수집·파싱·DB 코드를 같이 작성하고 실행 | 1–3 |
| **문서 판독기** | 공시 원문을 읽고 텀시트를 구조화 데이터로 변환·검증 | 2 |
| **분석가** | 로컬 MCP 서버로 연결한 상품 DB·벡터 검색을 직접 호출해 근거 있는 답변 생성 | 4–7 |

임베딩과 벡터 DB는 모두 로컬(무료)에서 돌아가므로 수강생은 API 키가 필요 없습니다.

## 코스 흐름

```
EDGAR (424B2·FWP)
   │  모듈 1: 수집
   ▼
원문 HTML / 평문 텍스트
   │  모듈 2: AI 정보 추출 (텀시트 → JSON)
   ▼
상품 DB (DuckDB)  ──────────┐
   │  모듈 3: DB 설계·검증     │
   ▼                          │
청크 + 임베딩 (Chroma)        │
   │  모듈 4–5: RAG·평가       │
   ▼                          ▼
Claude Desktop ◀── MCP 서버 (SQL 질의 + 벡터 검색)
   │  모듈 6: JPM 상품군 분석
   │  모듈 7: 신상품 설계·프라이싱
   ▼
모듈 8: 캡스톤 – 신상품 제안서
```

## 왜 JP Morgan 구조화 노트인가

* 발행 법인: **JPMorgan Chase Financial Company LLC** (CIK 1665650), 보증: **JPMorgan Chase & Co.** (CIK 19617)
* 미국 공모 구조화 노트는 상품마다 SEC에 **가격확정 보충설명서(424B2)**를 제출합니다. 기초자산, 쿠폰, 배리어, 조기상환 일정, **JPM이 직접 산정한 추정가치(estimated value)**까지 공개됩니다.
* 최근 1년간 이 법인 하나가 낸 424B2만 **2만 건 이상**입니다. 경쟁사의 상품 전략을 정량적으로 들여다볼 수 있는 드문 데이터입니다.

### EDGAR 공시 유형

| 양식 | 의미 | 코스에서의 쓰임 |
|---|---|---|
| **424B2** | 가격 확정 후 최종 보충설명서 (Pricing Supplement) | **주 데이터**: 최종 조건·발행금액·추정가치 |
| **FWP** | Free Writing Prospectus, 판매용 예비 텀시트 | 마케팅 문구·예비 조건 (보조) |
| **424B3** | 기본 설명서·상품 보충설명서 (Product Supplement) | 상품 유형별 일반 조건 정의 (RAG 참고 문서) |

## 424B2 해부

대부분의 JPM 가격확정 보충설명서는 다음 순서를 따릅니다. 모듈 2에서 이 구조를 이용해 정보를 추출합니다.

1. **표지**: 발행금액, 상품명(예: *Auto Callable Contingent Interest Notes Linked to …*), 핵심 요약 bullet
2. **Key Terms**: 기초자산, 가격결정일·발행일·만기일, 쿠폰율, 배리어/버퍼, 조기상환 조건, CUSIP
3. **Hypothetical Payout / Examples**: 가상 시나리오별 상환액
4. **Selected Risk Considerations**: 위험 요인
5. **The Underlying(s)**: 기초자산 설명·과거 성과
6. **Estimated Value**: 발행가 대비 JPM 추정가치 → **발행사 마진의 대리변수**
7. **Supplemental Plan of Distribution**: 판매 수수료 등

### 추출할 데이터 사전 (초안)

| 필드 | 예시 |
|---|---|
| `product_type` | autocallable_contingent, buffered_return_enhanced, trigger_plus, digital … |
| `underlyings` | `[{"name": "S&P 500", "ticker": "SPX", "weight": 1.0}]` |
| `worst_of` | true/false |
| `pricing_date`, `issue_date`, `maturity_date`, `tenor_years` | 2026-11-06, … , 6.0 |
| `principal_amount` | 775000 |
| `coupon_rate_pa`, `coupon_frequency`, `coupon_barrier_pct`, `memory_coupon` | 0.0925, monthly, 0.70, true |
| `autocall_frequency`, `autocall_trigger_pct`, `first_call_date` | quarterly, 1.00, 2027-11-08 |
| `downside_type`, `barrier_pct`, `buffer_pct` | barrier, 0.70, null |
| `upside_participation`, `cap_pct` | 1.5, 0.25 |
| `estimated_value` | 950.30 (per $1,000) |
| `cusip` | 48136… |

## Claude Desktop 사용법

1. Claude Desktop → **Code** 탭 → 폴더로 이 저장소(`AIFinance`)를 엽니다.
2. 각 노트북의 **🤖 Claude 실습** 상자에 있는 프롬프트를 Code 탭에 그대로 붙여 넣습니다.
3. Claude가 코드를 작성하고 실행하면 결과를 **반드시 원문과 대조해 검증**합니다. 실무에서 가장 중요한 습관입니다.

> 💡 좋은 프롬프트의 조건: **목표 + 입력 위치 + 출력 형식 + 검증 방법**을 함께 적는다.

## 환경 설정

터미널에서 한 번만 실행합니다. ([uv](https://docs.astral.sh/uv/) 필요)

```bash
uv sync                  # 가상환경 생성 + 패키지 설치
cp .env.example .env     # SEC 연락처 입력 (이름 + 이메일)
uv run jupyter lab       # 노트북 실행
```

아래 셀로 설정을 확인합니다.

In [ ]:
import sys
from aifinance import EdgarClient, JPM_FINANCIAL_CIK, DATA_DIR
from aifinance.config import sec_user_agent

print("Python:", sys.version.split()[0])
print("데이터 폴더:", DATA_DIR)
sec_user_agent()  # 설정이 없으면 여기서 안내 메시지와 함께 오류가 납니다
print("SEC_USER_AGENT 설정 OK")

In [ ]:
# EDGAR 연결 테스트: 발행사 이름과 최근 공시 건수
client = EdgarClient()
filings = client.list_filings(JPM_FINANCIAL_CIK)
print(f"최근 공시 {len(filings):,}건, 기간 {filings.filingDate.min():%Y-%m-%d} ~ {filings.filingDate.max():%Y-%m-%d}")
filings.head()

### 🤖 Claude 실습 0

> 이 저장소의 `src/aifinance/edgar.py`를 읽고, SEC 접근 규칙(User-Agent, 초당 요청 수)을 어떻게 지키고 있는지 설명해줘. 그리고 개선할 점이 있으면 알려줘.